In [3]:
import pandas as pd
import numpy as np
import os

# --- تنظیمات آدرس‌ها ---
file_path = r'second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx'
output_filename = r'outputs\G11\dsas_g11_bearings_vibration_temp_univariate\univariate\dsas_g11_univariate_output3.xlsx'

# سنسورهای هدف
target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']

def run_ewma_analysis():
    if not os.path.exists(file_path):
        print(f"❌ خطا: فایل در مسیر زیر یافت نشد:\n{file_path}")
        return

    try:
        # ۱. بارگذاری و مرتب‌سازی داده‌ها بر اساس زمان
        df = pd.read_excel(file_path, parse_dates=['date'])
        df = df.sort_values('date')
        print("✅ مرحله ۱: فایل با موفقیت بارگذاری شد.")
    except Exception as e:
        print(f"❌ خطا در خواندن اکسل: {e}")
        return

    # ۲. جداسازی بازه یک ماه اخیر (Fault Period)
    last_date = df['date'].max()
    one_month_ago = last_date - pd.Timedelta(days=30)
    df_recent = df[df['date'] >= one_month_ago].copy()

    # ۳. محاسبه EWMA و تجمیع نتایج
    ewma_results_list = []

    print("⏳ در حال محاسبه EWMA (ضریب 0.2) برای سنسورهای هدف...")

    for col in target_sensors:
        if col in df_recent.columns:
            # ایجاد دیتافریم موقت برای هر سنسور
            temp_df = df_recent[['date', col]].copy()

            # تغییر نام ستون اصلی به Value
            temp_df = temp_df.rename(columns={col: 'Value'})

            # افزودن ستون نام سنسور
            temp_df['AssetID'] = col

            # محاسبه مقدار EWMA طبق فرمول Et = 0.2  Xt + 0.8  Et-1
            # پارامتر adjust=False برای رعایت دقیق فرمول بازگشتی ضروری است
            temp_df['EWMA'] = temp_df['Value'].ewm(alpha=0.2, adjust=False).mean()

            # مرتب‌سازی ستون‌ها طبق درخواست شما
            final_temp = temp_df[['date', 'AssetID', 'Value', 'EWMA']]
            ewma_results_list.append(final_temp)

    if not ewma_results_list:
        print("⚠️ هشدار: هیچ‌کدام از سنسورهای هدف در فایل ورودی پیدا نشدند.")
        return

    # ۴. یکی کردن نتایج تمام سنسورها
    df_final_output = pd.concat(ewma_results_list, ignore_index=True)

    # ۵. ذخیره خروجی نهایی
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        df_final_output.to_excel(output_filename, index=False, sheet_name='EWMA_Comparison')
        print(f"🚀 گزارش نهایی با موفقیت ساخته شد. \nمحل ذخیره: {output_filename}")
    except Exception as e:
        print(f"❌ خطا در ذخیره فایل اکسل: {e}")

# اجرای تابع
run_ewma_analysis()

✅ مرحله ۱: فایل با موفقیت بارگذاری شد.
⏳ در حال محاسبه EWMA (ضریب 0.2) برای سنسورهای هدف...
🚀 گزارش نهایی با موفقیت ساخته شد. 
محل ذخیره: outputs\G11\dsas_g11_bearings_vibration_temp_univariate\univariate\dsas_g11_univariate_output3.xlsx
